# GSM 2G Security — A3 / A8 (COMP128-1) and A5/1

This notebook breaks down our previously tested **`gsm_a3_a8_a5_complete.py`** script into individual steps. It is a local educational simulation of historical GSM cryptographic algorithms; it **does not connect to a live GSM network** and must not be used for modern secure encryption.

**Important:** A3/A8 produce SRES and Kc; A5/1 uses Kc + COUNT to generate a keystream. Run code cells **in order, from top to bottom**.

## 0 — Environment and Imports

Select `C:\GSM_Security_Algorithms\.venv\Scripts\python.exe` as the kernel in VS Code/Jupyter. The `comp128` package was already installed in this environment. This cell displays the Python version and active interpreter.

In [19]:
import sys
import secrets
from comp128.comp128v1 import Comp128v1

print("Python:", sys.version.split()[0])
print("Interpreter:", sys.executable)
print("COMP128-1 import OK")

Python: 3.12.10
Interpreter: c:\Users\Elif\raytracing\.venv\Scripts\python.exe
COMP128-1 import OK


## 1 — A3 and A8: The Real COMP128-1 Function

**Inputs:** `Ki` (16 bytes) and `RAND` (16 bytes). **Outputs:** `SRES` (4 bytes) and `Kc` (8 bytes). COMP128-1 forces the last 10 bits of Kc to zero. In a real network, `Ki` must remain secret.

In [20]:
def a3_a8(ki: bytes, rand: bytes) -> tuple[bytes, bytes]:
    """Compute actual historic COMP128-1 GSM authentication triplet outputs."""
    if len(ki) != 16 or len(rand) != 16:
        raise ValueError('Ki and RAND must both be exactly 16 bytes (128 bits).')
    sres, kc = Comp128v1().comp128v1(bytearray(ki), bytearray(rand))
    sres, kc = bytes(sres), bytes(kc)
    assert len(sres) == 4 and len(kc) == 8
    assert kc[-1] == 0 and (kc[-2] & 0b11) == 0  # COMP128-1 zeroes last 10 bits
    return sres, kc

### What `a3_a8` does

```
   Ki  (128 bit)                 RAND  (128 bit)
   secret, on SIM                public, from network
        │                              │
        └──────────────┬───────────────┘
                       ▼
             ┌───────────────────┐
             │     COMP128-1     │   same calculation on SIM and AuC
             └─────────┬─────────┘
            ┌──────────┴──────────┐
            ▼ A3                  ▼ A8
      SRES (32 bit)          Kc (64 bit)
      → sent to network      → key for A5 cipher

  Kc:  [ 54 real key bits                         ][ 0000000000 ]
                                                     └ last 10 bits always 0 → 2^54, not 2^64
```


## 2 — A5/1 Register Definitions

A5/1 has three LFSRs: R1 is **19 bits**, R2 is **22 bits**, and R3 is **23 bits**. `taps` identifies the bit positions used to calculate feedback. The last number in each specification is the majority-clocking bit position. Bit indices are **zero-based**.

In [21]:
REGISTER_SPECS = ((19, (13, 16, 17, 18), 8),
                  (22, (20, 21), 10),
                  (23, (7, 20, 21, 22), 10))

print("(length, feedback taps, clock bit):")
for i,spec in enumerate(REGISTER_SPECS, start=1):
    print(f"R{i}: {spec}")

(length, feedback taps, clock bit):
R1: (19, (13, 16, 17, 18), 8)
R2: (22, (20, 21), 10)
R3: (23, (7, 20, 21, 22), 10)


### The three A5/1 registers

```
            bit 0 ─────────── shift → ──────────► MSB
  R1 (19)   ·······C····t··ttt.     ──┐
  R2 (22)   ·········C·········tt.  ──┼── ⊕ ──► output bit
  R3 (23)   ······t··C·········ttt. ──┘

  C = clock bit (8, 10, 10)
  t = feedback tap → XOR of taps becomes the new bit 0
  Output = XOR of the three MSBs (bits 18, 21, 22)
```

**Majority clocking:** look at the three C bits, find the majority, and shift only the registers that agree with it.

```
  C bits:  R1 = 1   R2 = 0   R3 = 1   → majority = 1
           R1 shifts, R3 shifts, R2 waits
```

At every step 2 or 3 registers move, never only one. This irregular movement is what makes A5/1 nonlinear.

## 3 — A5/1: Shifting One Register

The selected feedback bits are XORed. The register shifts toward higher bit positions, and the new feedback bit is inserted at bit position 0. Bits beyond the register length are discarded.

In [22]:
def shift_register(state: int, length: int, taps: tuple[int, ...]) -> int:
    """Shift towards higher bit positions and XOR feedback into bit 0."""
    feedback = 0
    for tap in taps:
        feedback ^= (state >> tap) & 1
    return ((state << 1) | feedback) & ((1 << length) - 1)

## 4 — A5/1 Class: Clocking and Output Bit

`clock_all()` is used during Kc/COUNT initialization; `clock_majority()` is used during warm-up and keystream generation. `output_bit()` XORs the most significant bits of the three registers.

In [23]:
class A51:
    """Three real A5/1 LFSRs with irregular majority clocking."""

    def __init__(self):
        self.regs = [0, 0, 0]
# First 64 steps: Shift all three LFSRs and XOR each Kc bit into them.
# Next 22 steps: Shift all three LFSRs and XOR each COUNT bit into them.
# Next 100 steps: Use clock_majority() instead of clock_all().
# During this phase, no additional input bits are added.

    def clock_all(self):
        """Regular clocking used only for key/frame initialization."""
        for i, (length, taps, _) in enumerate(REGISTER_SPECS):
            self.regs[i] = shift_register(self.regs[i], length, taps)

    def clock_majority(self):
        """Clock registers whose clock bit matches the majority of three bits."""
        clock_bits = [(self.regs[i] >> spec[2]) & 1
                      for i, spec in enumerate(REGISTER_SPECS)]
        majority = int(sum(clock_bits) >= 2)
        for i, (length, taps, _) in enumerate(REGISTER_SPECS):
            if clock_bits[i] == majority:
                self.regs[i] = shift_register(self.regs[i], length, taps)

    def mix_in(self, bit):
        for i in range(3):
            self.regs[i] ^= bit

    def output_bit(self):
        return ((self.regs[0] >> 18) ^
                (self.regs[1] >> 21) ^
                (self.regs[2] >> 22)) & 1

<h3>A5/1 — Majority Clocking, Feedback & Shifting</h3>

<div style=“font-family:monospace;overflow-x:auto;“>

<h4 style=“color:#61AFEF;“>
STEP 1 — REGISTER STATES BEFORE CLOCKING
</h4>

<div style=“margin:15px 0;color:#ddd;“>
<b>R1 — 19 bits</b>
</div>

<table style=“border-collapse:collapse;text-align:center;font-family:monospace;“>
<tr>
<td>Index</td>
<td>18</td><td>17</td><td>16</td><td>15</td><td>14</td>
<td>13</td><td>12</td><td>11</td><td>10</td><td>9</td>
<td style=“color:#E06C75”>8</td>
<td>7</td><td>6</td><td>5</td><td>4</td><td>3</td>
<td>2</td><td>1</td><td>0</td>
</tr>
<tr>
<td>Before</td>
<td>1</td><td>1</td><td>0</td><td>0</td><td>1</td>
<td>1</td><td>0</td><td>0</td><td>1</td><td>0</td>
<td style=“background:#713F45;color:white”><b>1</b></td>
<td>0</td><td>1</td><td>0</td><td>0</td><td>1</td>
<td>1</td><td>0</td><td>1</td>
</tr>
</table>
<div style=“color:#E06C75;margin:8px 0;“>
Clock bit: R1[8] = 1
</div>

<div style=“margin:25px 0 10px;color:#ddd;“>
<b>R2 — 22 bits</b>
</div>

<table style=“border-collapse:collapse;text-align:center;font-family:monospace;“>
<tr>
<td>Index</td>
<td>21</td><td>20</td><td>19</td><td>18</td><td>17</td>
<td>16</td><td>15</td><td>14</td><td>13</td><td>12</td>
<td>11</td><td style=“color:#E06C75”>10</td>
<td>9</td><td>8</td><td>7</td><td>6</td><td>5</td>
<td>4</td><td>3</td><td>2</td><td>1</td><td>0</td>
</tr>
<tr>
<td>Before</td>
<td>1</td><td>1</td><td>0</td><td>0</td><td>1</td>
<td>1</td><td>0</td><td>0</td><td>1</td><td>1</td>
<td>0</td><td style=“background:#713F45;color:white”><b>0</b></td>
<td>1</td><td>1</td><td>0</td><td>0</td><td>1</td>
<td>1</td><td>0</td><td>0</td><td>0</td><td>1</td>
</tr>
</table>
<div style=“color:#E06C75;margin:8px 0;“>
Clock bit: R2[10] = 0
</div>

<div style=“margin:25px 0 10px;color:#ddd;“>
<b>R3 — 23 bits</b>
</div>

<table style=“border-collapse:collapse;text-align:center;font-family:monospace;“>
<tr>
<td>Index</td>
<td>22</td><td>21</td><td>20</td><td>19</td><td>18</td>
<td>17</td><td>16</td><td>15</td><td>14</td><td>13</td>
<td>12</td><td>11</td><td style=“color:#E06C75”>10</td>
<td>9</td><td>8</td><td>7</td><td>6</td><td>5</td>
<td>4</td><td>3</td><td>2</td><td>1</td><td>0</td>
</tr>
<tr>
<td>Before</td>
<td>0</td><td>0</td><td>0</td><td>1</td><td>1</td>
<td>0</td><td>1</td><td>0</td><td>0</td><td>1</td>
<td>0</td><td>1</td><td style=“background:#713F45;color:white”><b>1</b></td>
<td>0</td><td>1</td><td>0</td><td>1</td><td>0</td>
<td>0</td><td>1</td><td>1</td><td>0</td><td>1</td>
</tr>
</table>
<div style=“color:#E06C75;margin:8px 0;“>
Clock bit: R3[10] = 1
</div>

</div>

<div style=“font-family:monospace;overflow-x:auto;color:#ddd;“>

<h4 style=“color:#61AFEF;margin-top:30px;“>
STEP 2 — MAJORITY CLOCKING
</h4>

<table style=“border-collapse:collapse;text-align:center;“>
<tr style=“color:#61AFEF;“>
<th style=“padding:8px 18px;“>Register</th>
<th style=“padding:8px 18px;“>Clock bit</th>
<th style=“padding:8px 18px;“>Value</th>
<th style=“padding:8px 18px;“>Decision</th>
</tr>
<tr>
<td>R1</td><td>8</td>
<td style=“background:#713F45;color:white;“>1</td>
<td style=“color:#98C379;“>SHIFT</td>
</tr>
<tr>
<td>R2</td><td>10</td>
<td style=“background:#713F45;color:white;“>0</td>
<td style=“color:#E06C75;“>HOLD</td>
</tr>
<tr>
<td>R3</td><td>10</td>
<td style=“background:#713F45;color:white;“>1</td>
<td style=“color:#98C379;“>SHIFT</td>
</tr>
</table>

<div style=“text-align:center;margin:18px 0;color:#E5C07B;font-weight:bold;“>
MAJORITY(1, 0, 1) = 1
</div>

<div style=“text-align:center;color:#98C379;“>
R1 SHIFTS &nbsp; | &nbsp;
<span style=“color:#E06C75;“>R2 HOLDS</span>
&nbsp; | &nbsp; R3 SHIFTS
</div>

<h4 style=“color:#61AFEF;margin-top:35px;“>
STEP 3 — FEEDBACK AND SHIFTING
</h4>

<div style=“color:#E5C07B;margin:15px 0;“>
R1 FEEDBACK — taps [13, 16, 17, 18]
</div>

<div style=“color:#E5C07B;“>
1 ⊕ 0 ⊕ 1 ⊕ 1 = 1
</div>

<div style=“color:#98C379;margin:12px 0;“>
↓ SHIFT LEFT — New bit 0 = 1
</div>

<table style=“border-collapse:collapse;text-align:center;font-family:monospace;“>
<tr>
<td>Index</td>
<td>18</td><td>17</td><td>16</td><td>15</td><td>14</td>
<td>13</td><td>12</td><td>11</td><td>10</td><td>9</td>
<td>8</td><td>7</td><td>6</td><td>5</td><td>4</td>
<td>3</td><td>2</td><td>1</td><td>0</td>
</tr>
<tr>
<td style=“color:#98C379;“>After</td>
<td>1</td><td>0</td><td>0</td><td>1</td><td>1</td>
<td>0</td><td>0</td><td>1</td><td>0</td><td>1</td>
<td>0</td><td>1</td><td>0</td><td>0</td><td>1</td>
<td>1</td><td>0</td><td>1</td>
<td style=“background:#234d40;color:#98C379;“><b>1</b></td>
</tr>
</table>

<div style=“color:#E06C75;margin:25px 0 10px;“>
R2 — HOLD (NO SHIFT)
</div>

<table style=“border-collapse:collapse;text-align:center;font-family:monospace;“>
<tr>
<td>Index</td>
<td>21</td><td>20</td><td>19</td><td>18</td><td>17</td>
<td>16</td><td>15</td><td>14</td><td>13</td><td>12</td>
<td>11</td><td>10</td><td>9</td><td>8</td><td>7</td>
<td>6</td><td>5</td><td>4</td><td>3</td><td>2</td>
<td>1</td><td>0</td>
</tr>
<tr>
<td style=“color:#E06C75;“>After</td>
<td>1</td><td>1</td><td>0</td><td>0</td><td>1</td>
<td>1</td><td>0</td><td>0</td><td>1</td><td>1</td>
<td>0</td><td>0</td><td>1</td><td>1</td><td>0</td>
<td>0</td><td>1</td><td>1</td><td>0</td><td>0</td>
<td>0</td><td>1</td>
</tr>
</table>

<div style=“color:#E5C07B;margin:25px 0 10px;“>
R3 FEEDBACK — taps [7, 20, 21, 22]
</div>

<div style=“color:#E5C07B;“>
0 ⊕ 0 ⊕ 0 ⊕ 0 = 0
</div>

<div style=“color:#98C379;margin:12px 0;“>
↓ SHIFT LEFT — New bit 0 = 0
</div>

<table style=“border-collapse:collapse;text-align:center;font-family:monospace;“>
<tr>
<td>Index</td>
<td>22</td><td>21</td><td>20</td><td>19</td><td>18</td>
<td>17</td><td>16</td><td>15</td><td>14</td><td>13</td>
<td>12</td><td>11</td><td>10</td><td>9</td><td>8</td>
<td>7</td><td>6</td><td>5</td><td>4</td><td>3</td>
<td>2</td><td>1</td><td>0</td>
</tr>
<tr>
<td style=“color:#98C379;“>After</td>
<td>0</td><td>0</td><td>1</td><td>1</td><td>0</td>
<td>1</td><td>0</td><td>0</td><td>1</td><td>0</td>
<td>1</td><td>1</td><td>0</td><td>1</td><td>0</td>
<td>1</td><td>0</td><td>0</td><td>1</td><td>1</td>
<td>0</td><td>1</td>
<td style=“background:#234d40;color:#98C379;“><b>0</b></td>
</tr>
</table>

<h4 style=“color:#61AFEF;margin-top:35px;“>
STEP 4 — KEYSTREAM OUTPUT
</h4>

<div style=“text-align:center;margin:20px 0;color:#E5C07B;“>
R1[18] ⊕ R2[21] ⊕ R3[22]
</div>

<div style=“text-align:center;font-size:20px;color:#ddd;“>
1 ⊕ 1 ⊕ 0
</div>

<div style=“text-align:center;color:#98C379;font-size:22px;margin:15px;“>
↓
</div>

<div style=“text-align:center;color:#98C379;font-size:20px;font-weight:bold;“>
KEYSTREAM BIT = 0
</div>

<div style=“margin-top:25px;text-align:center;color:#aaa;font-size:12px;“>
<span style=“color:#E06C75;“>■</span> Clock bit &nbsp;
<span style=“color:#E5C07B;“>■</span> Feedback &nbsp;
<span style=“color:#98C379;“>■</span> Shifted state
</div>

</div>

## 5 — A5/1: Load Kc and COUNT, Then Run 100 Warm-up Clocks

All three registers start at zero. The 64 bits of Kc and then the 22 bits of COUNT are mixed into the internal state. The algorithm then performs **100 majority-clocking steps**; the outputs from this phase are discarded.

In [24]:
def a51_initialize(kc: bytes, count: int, verbose: bool = False) -> A51:
    """64 key bits + 22 COUNT bits + 100 majority-clock warm-up steps."""
    if len(kc) != 8:
        raise ValueError('Kc must be 8 bytes (64 bits).')
    if not 0 <= count < (1 << 22):
        raise ValueError('GSM COUNT must be a 22-bit unsigned integer.')
    s = A51()
    # Feed key LSB first, from first byte to last (Briceno convention).
    for i in range(64):
        s.clock_all()
        s.mix_in((kc[i // 8] >> (i % 8)) & 1)
    if verbose:
        print('  After loading 64 Kc bits:', [hex(r) for r in s.regs])
    for i in range(22):
        s.clock_all()
        s.mix_in((count >> i) & 1)
    if verbose:
        print('  After loading 22 COUNT bits:', [hex(r) for r in s.regs])
    for _ in range(100):
        s.clock_majority()
    if verbose:
        print('  After 100 warm-up clocks:', [hex(r) for r in s.regs])
    return s

## 6 — A5/1: Generate 228 Keystream Bits

After each majority clock, one bit is produced. The first **114 bits are for downlink** and the next **114 bits are for uplink**.

In [25]:
def a51_keystream_bits(kc: bytes, count: int, verbose: bool = False):
    """Return first 114 downlink and next 114 uplink bits for one COUNT."""
    s = a51_initialize(kc, count, verbose)
    bits = []
    for _ in range(228):
        s.clock_majority()
        bits.append(s.output_bit())
    return bits[:114], bits[114:]

## 7 — Helper Functions

Convert bytes to bits, pack a 114-bit burst, convert a GSM TDMA frame number to the 22-bit COUNT value, and XOR equal-length bit sequences.

In [26]:
def bits_from_bytes(data: bytes) -> list[int]:
    """Most significant bit first, suitable for displaying/packing bursts."""
    return [(b >> (7 - j)) & 1 for b in data for j in range(8)]

def bytes_from_burst(bits: list[int]) -> bytes:
    """Pack 114 bits into 15 bytes; unused last six bits are zero."""
    if len(bits) != 114:
        raise ValueError('A GSM A5/1 cipher burst is 114 bits.')
    out = bytearray(15)
    for i, b in enumerate(bits):
        out[i // 8] |= b << (7 - (i % 8))
    return bytes(out)

def gsm_frame_to_count(fn: int) -> int:
    """Convert GSM TDMA frame number FN into 22-bit COUNT (T1/T3/T2)."""
    if not 0 <= fn < 26 * 51 * 2048:
        raise ValueError('FN outside GSM hyperframe range.')
    t1 = fn // (26 * 51)
    t2 = fn % 26
    t3 = fn % 51
    return (t1 << 11) | (t3 << 5) | t2

def xor_bits(a: list[int], b: list[int]) -> list[int]:
    if len(a) != len(b):
        raise ValueError('XOR inputs must be the same length.')
    return [x ^ y for x, y in zip(a, b)]

## 8 — Complete Demonstration in One Cell (Optional)

The following function runs all the steps from the original `.py` script in sequence. The preceding cells already show each operation individually, so this is **optional**.

In [27]:
def demonstration():
    print('\n========== GSM 2G AUTH + CIPHER DEMONSTRATION ==========')
    # Demo secret (actual SIM Ki is provisioned, not normally randomly created per run).
    ki_sim = bytes.fromhex('00112233445566778899AABBCCDDEEFF')
    ki_operator = ki_sim  # network AuC holds the same provisioned secret
    rand = secrets.token_bytes(16)  # challenge sent from network to SIM
    print('1. RAND (public)                  :', rand.hex())
    print('   Ki (DEMO ONLY - normally secret):', ki_sim.hex())

    # Network computes SRES and Kc. It gives RAND to the SIM and retains its values.
    expected_sres, kc_operator = a3_a8(ki_operator, rand)
    sres_sim, kc_sim = a3_a8(ki_sim, rand)
    print('2. SIM SRES (A3)                 :', sres_sim.hex())
    print('   Network expected SRES         :', expected_sres.hex())
    authenticated = secrets.compare_digest(sres_sim, expected_sres)
    print('3. Subscriber authenticated      :', authenticated)
    if not authenticated:
        raise RuntimeError('Authentication failed. No ciphering started.')
    assert kc_sim == kc_operator
    print('4. Kc generated by A8            :', kc_sim.hex())
    print('   Kc last 10 bits zero (COMP128-1):', (kc_sim[-2] & 3) == 0 and kc_sim[-1] == 0)

    # A5/1 is run independently from COMP128-1. Use a sample GSM TDMA frame number.
    fn = 123456
    count = gsm_frame_to_count(fn)
    print('5. TDMA FN                       :', fn)
    print('   22-bit COUNT for A5/1          :', hex(count))
    downlink, uplink = a51_keystream_bits(kc_sim, count, verbose=True)
    print('6. Downlink keystream (114 bits)  :', ''.join(map(str, downlink)))
    print('   Uplink keystream (114 bits)    :', ''.join(map(str, uplink)))

    # Illustrative 114-bit payload (NOT actual GSM speech/channel-coded burst).
    message = b'HELLO GSM'  # 9 bytes = 72 bits
    plaintext_bits = bits_from_bytes(message) + [0] * (114 - len(message) * 8)
    cipher_bits = xor_bits(plaintext_bits, downlink)
    received_bits = xor_bits(cipher_bits, downlink)
    recovered = bytes_from_burst(received_bits)[:len(message)]
    print('7. Original toy 114-bit payload  :', ''.join(map(str, plaintext_bits)))
    print('8. Ciphertext (XOR with downlink):', ''.join(map(str, cipher_bits)))
    print('9. Decrypted payload             :', recovered.decode('ascii'))
    assert recovered == message
    print('\nPASS: subscriber authentication and local cipher/decrypt round trip.')
    print('This does NOT transmit over real GSM or implement a complete GSM stack.')

# İstersen aşağıdaki satırın başındaki # işaretini kaldır:
demonstration()


========== GSM 2G AUTH + CIPHER DEMONSTRATION ==========
1. RAND (public)                  : c8021383866746c2a36a86854a9b304a
   Ki (DEMO ONLY - normally secret): 00112233445566778899aabbccddeeff
2. SIM SRES (A3)                 : 95af708d
   Network expected SRES         : 95af708d
3. Subscriber authenticated      : True
4. Kc generated by A8            : 6af9ca4d5b443400
   Kc last 10 bits zero (COMP128-1): True
5. TDMA FN                       : 123456
   22-bit COUNT for A5/1          : 0x2ec88
  After loading 64 Kc bits: ['0x50f16', '0x21bf98', '0x76fb0e']
  After loading 22 COUNT bits: ['0x23087', '0x268d79', '0x279ade']
  After 100 warm-up clocks: ['0x70b90', '0xd09cb', '0x3137c']
6. Downlink keystream (114 bits)  : 111101100000011011001101011101101110100001011101001100000010101110011111001100110110100001110011000100010010101011
   Uplink keystream (114 bits)    : 001100011010111000111000111010010101100110011110011111000000011101001011100110010001000110100101000000001101110011


---
### Study Notes

**A3:** `SRES` is the first 4 bytes of the COMP128-1 output from `Ki` and `RAND`. **A8:** `Kc` is the next 8 bytes. **A5/1:** `Kc + COUNT → R1/R2/R3 → 100 warm-up clocks → 228-bit keystream`.

Note: COMP128-1 and A5/1 are historically vulnerable algorithms. This notebook is intended **for research and education only**.